# Stage 2 – Fraud Decision Engine

This notebook implements Stage 2 of the fraud detection pipeline.
It operates only on transactions flagged by Stage 1 and focuses on
high-precision fraud confirmation and final decisioning.


In [27]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

In [2]:
df =  pd.read_csv("C:\\Users\\hp\\Downloads\\archive (1)\\main.csv")

In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


### loading stage one model ###

In [4]:
rf_stage1 = joblib.load('stage1fraud.pkl')

In [5]:
import os
os.getcwd()


'C:\\Users\\hp\\OneDrive\\Desktop\\fraud-detection-project\\notebooks'

### APPLYING stage 1 model to flag transactions ###


In [5]:
df = pd.get_dummies(df, columns=['type'], drop_first=True)


In [7]:
df.head()

,step,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,False,False,True,False
1,1,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,False,False,True,False
2,1,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,False,False,False,True
3,1,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,True,False,False,False
4,1,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,False,False,True,False


In [43]:
rf_stage1.feature_names_in_

array(['step', 'amount', 'oldbalanceOrg', 'type_CASH_OUT', 'type_DEBIT',
       'type_PAYMENT', 'type_TRANSFER'], dtype=object)

In [8]:
type_cols = [
    'type_CASH_OUT',
    'type_DEBIT',
    'type_PAYMENT',
    'type_TRANSFER' 
]
df[type_cols] = df[type_cols].astype(int)

In [9]:
df.head()

,step,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,0,0,1,0
1,1,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,0,0,1,0
2,1,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,0,0,0,1
3,1,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,1,0,0,0
4,1,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,0,0,1,0


### Takes the column the model knows ###

In [10]:
X_clean = df[rf_stage1.feature_names_in_]

In [11]:
X_clean.head()

,step,amount,oldbalanceOrg,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,0,0,1,0
1,1,1864.28,21249.0,0,0,1,0
2,1,181.00,181.0,0,0,0,1
3,1,181.00,181.0,1,0,0,0
4,1,11668.14,41554.0,0,0,1,0


In [19]:
STAGE1_THRESHOLD = 0.06

def stage1_decision(model, X):
    proba = model.predict_proba(X)[:, 1]
    return np.where(proba >= STAGE1_THRESHOLD, "FLAG", "APPROVE")

# Outside the function
df['clean_flag'] = stage1_decision(rf_stage1, X_clean)
df['clean_flag'].value_counts()


clean_flag
APPROVE    5435275
FLAG        927345
Name: count, dtype: int64

In [20]:
df_flagged = df[df['clean_flag'] == "FLAG"].copy()


In [21]:
# How much the origin account changed
df_flagged['orig_balance_change'] = df_flagged['oldbalanceOrg'] - df_flagged['newbalanceOrig']

# How much the destination account changed
df_flagged['dest_balance_change'] = df_flagged['newbalanceDest'] - df_flagged['oldbalanceDest']

# Check if origin account went to zero
df_flagged['is_zero_orig_after'] = (df_flagged['newbalanceOrig'] == 0).astype(int)

# Optionally, ratio of amount to old balance (new feature)
df_flagged['amount_to_old_balance_ratio'] = df_flagged['amount'] / (df_flagged['oldbalanceOrg'] + 1)


In [22]:

X_stage2 = df_flagged[['amount', 'orig_balance_change', 'dest_balance_change',
                       'is_zero_orig_after', 'amount_to_old_balance_ratio']]

y_stage2 = df_flagged['isFraud']  # target


In [72]:
def check_amount_to_balance_ratio(X):
    """
    Failure detection for 'amount_to_old_balance_ratio'.
    Returns True if the feature is healthy, False if a failure is detected.
    """
    feature = 'amount_to_old_balance_ratio'
    
    # 1. Missing column
    if feature not in X.columns:
        print(f"FAILURE DETECTED: {feature} is missing")
        return False
    
    # 2. Constant or all zeros
    if X[feature].nunique() <= 1:
        print(f"FAILURE DETECTED: {feature} is constant or all identical")
        return False
    
    # 3. Out-of-range values
    if (X[feature] < 0.0).any() or (X[feature] > 6500000).any():  # example upper bound
        print(f"FAILURE DETECTED: {feature} has values outside expected range")
        return False
    
    # Passed all checks
    return True


# Example usage before Stage-2 inference
if not check_amount_to_balance_ratio(X_stage2):
    # Explicit fallback policy
    print("Triggering fallback: BLOCK disabled, route to REVIEW")
    df_flagged['stage2_action'] = df_flagged['prob_full'].apply(
        lambda p: "REVIEW" if p >= 0.25 else "APPROVE"
    )
else:
    # Normal Stage-2 decision
    df_flagged['stage2_action'] = df_flagged['prob_full'].apply(stage2_tier)


KeyError: 'prob_full'

In [25]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_stage2, y_stage2, test_size=0.2, stratify=y_stage2, random_state=42
)


In [28]:
rf_stage2 = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf_stage2.fit(X_train2, y_train2)


RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=150,
                       n_jobs=-1, random_state=42)

In [29]:
y_pred2 = rf_stage2.predict(X_test2)

print(confusion_matrix(y_test2, y_pred2))
print(classification_report(y_test2, y_pred2))


[[183827      0]
 [     4   1638]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    183827
           1       1.00      1.00      1.00      1642

    accuracy                           1.00    185469
   macro avg       1.00      1.00      1.00    185469
weighted avg       1.00      1.00      1.00    185469



## Stress testing ##

In [30]:
X_noisy = X_stage2.copy()

### Inject balance noise ###

In [38]:
noise_level = 0.02

df_noisy = df_flagged.copy()

for col in ['newbalanceOrig', 'newbalanceDest']:
    noise = np.random.uniform(1 - noise_level, 1 + noise_level, size=len(df_noisy))
    df_noisy[col] *= noise


In [39]:
df_noisy['orig_balance_change'] = (
    df_noisy['oldbalanceOrg'] - df_noisy['newbalanceOrig']
)

df_noisy['dest_balance_change'] = (
    df_noisy['newbalanceDest'] - df_noisy['oldbalanceDest']
)

df_noisy['is_zero_orig_after'] = (df_noisy['newbalanceOrig'] == 0).astype(int)

df_noisy['amount_to_old_balance_ratio'] = (
    df_noisy['amount'] / (df_noisy['oldbalanceOrg'] + 1)
)


In [40]:
X_stage2_noisy = df_noisy[
    ['amount',
     'orig_balance_change',
     'dest_balance_change',
     'is_zero_orig_after',
     'amount_to_old_balance_ratio']
]


In [41]:
y_pred_noisy = rf_stage2.predict(X_stage2_noisy)
y_proba_noisy = rf_stage2.predict_proba(X_stage2_noisy)[:, 1]




In [42]:
y_pred_clean = rf_stage2.predict(X_stage2)


In [43]:
prediction_flip_rate = (y_pred_clean != y_pred_noisy).mean()
print("Prediction flip rate:", prediction_flip_rate)


Prediction flip rate: 1.4018515223568359e-05


In [44]:
from sklearn.metrics import confusion_matrix, classification_report

print("CONFUSION MATRIX — CLEAN DATA")
print(confusion_matrix(y_stage2, y_pred_clean))

print("\nCONFUSION MATRIX — NOISY DATA")
print(confusion_matrix(y_stage2, y_pred_noisy))


CONFUSION MATRIX — CLEAN DATA
[[919135      0]
 [    15   8195]]

CONFUSION MATRIX — NOISY DATA
[[919135      0]
 [    28   8182]]


In [45]:
print("\nCLASSIFICATION REPORT — CLEAN DATA")
print(classification_report(y_stage2, y_pred_clean))

print("\nCLASSIFICATION REPORT — NOISY DATA")
print(classification_report(y_stage2, y_pred_noisy))



CLASSIFICATION REPORT — CLEAN DATA
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    919135
           1       1.00      1.00      1.00      8210

    accuracy                           1.00    927345
   macro avg       1.00      1.00      1.00    927345
weighted avg       1.00      1.00      1.00    927345


CLASSIFICATION REPORT — NOISY DATA
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    919135
           1       1.00      1.00      1.00      8210

    accuracy                           1.00    927345
   macro avg       1.00      1.00      1.00    927345
weighted avg       1.00      1.00      1.00    927345



In [46]:
from sklearn.metrics import recall_score

print("Fraud recall clean:", recall_score(y_stage2, y_pred_clean))
print("Fraud recall noisy:", recall_score(y_stage2, y_pred_noisy))


Fraud recall clean: 0.9981729598051157
Fraud recall noisy: 0.9965895249695493


Under 2% balance perturbation, Stage-2 false negatives increased from 15 to 28 while false positives remained zero, indicating high robustness with minor sensitivity at the decision boundary.”

### I then decided to use tiers as it is th industry level approach ###

In [49]:
# Get Stage-2 probability
y_proba = rf_stage2.predict_proba(X_stage2_noisy)[:, 1]  # use noisy or clean data as needed

# Apply tiered decision
def stage2_tier(prob):
    if prob >= 0.7:
        return "BLOCK"
    elif prob >= 0.4:
        return "REVIEW"
    else:
        return "APPROVE"

df_flagged['stage2_action'] = [stage2_tier(p) for p in y_proba]

# Quick check
df_flagged['stage2_action'].value_counts()


stage2_action
APPROVE    919158
BLOCK        8166
REVIEW         21
Name: count, dtype: int64

In [48]:
def stage2_tier(prob):
    if prob >= 0.75:
        return "BLOCK"
    elif prob >= 0.25:
        return "REVIEW"
    else:
        return "APPROVE"


In [50]:
pd.crosstab(
    df_flagged['stage2_action'],
    df_flagged['isFraud'],
    rownames=['Decision'],
    colnames=['Actual']
)


Actual,0,1
Decision,,
APPROVE,919134,24
BLOCK,0,8166
REVIEW,1,20


The tiered decision policy cleanly separates high-confidence fraud for automatic blocking, routes borderline cases to manual review, and limits fraud leakage in approved transactions to a negligible rate.”

### FEATURE REMOVAL TEST ###

### Establish a baseline control ###

In [53]:
# Predict probabilities with all features present
proba_full = rf_stage2.predict_proba(X_stage2)[:, 1]

df_eval = df_flagged.copy()
df_eval['prob_full'] = proba_full
df_eval['decision_full'] = df_eval['prob_full'].apply(stage2_tier)

df_eval['decision_full'].value_counts()

decision_full
APPROVE    919138
BLOCK        8183
REVIEW         24
Name: count, dtype: int64

###  Remove one Feature by neutralizing ###

In [54]:
X_no_ratio = X_stage2.copy()
X_no_ratio['amount_to_old_balance_ratio'] = 0

In [55]:
proba_no_ratio = rf_stage2.predict_proba(X_no_ratio)[:, 1]

df_eval['prob_no_ratio'] = proba_no_ratio
df_eval['decision_no_ratio'] = df_eval['prob_no_ratio'].apply(stage2_tier)

In [56]:
pd.crosstab(
    df_eval['decision_full'],
    df_eval['decision_no_ratio'],
    rownames=['With feature'],
    colnames=['Feature removed']
)


Feature removed,APPROVE,BLOCK,REVIEW
With feature,,,
APPROVE,321551,160960,436627
BLOCK,0,5434,2749
REVIEW,0,2,22


In [57]:
pd.crosstab(
    df_eval['decision_no_ratio'],
    df_eval['isFraud'],
    rownames=['Decision (feature removed)'],
    colnames=['Actual']
)


Actual,0,1
Decision (feature removed),,
APPROVE,321551,0
BLOCK,160960,5436
REVIEW,436624,2774


### amount_to balance_ratio is doing a amassive uncertainity stabilization.Without it the model overestimates risk
 ### It is a safety critical feature ###

In [59]:
X_no_orig_change = X_stage2.copy()
X_no_orig_change['orig_balance_change'] = 0  # simulate failure


In [60]:
proba_no_orig_change = rf_stage2.predict_proba(X_no_orig_change)[:, 1]
df_eval['prob_no_orig_change'] = proba_no_orig_change


In [61]:
df_eval['decision_no_orig_change'] = df_eval['prob_no_orig_change'].apply(stage2_tier)


In [62]:
pd.crosstab(
    df_eval['decision_full'],   # baseline decision with all features
    df_eval['decision_no_orig_change'],  # decision after feature removal
    rownames=['With feature'],
    colnames=['Feature removed']
)


Feature removed,APPROVE,BLOCK,REVIEW
With feature,,,
APPROVE,919138,0,0
BLOCK,1,8182,0
REVIEW,5,0,19


orig_balance_change is helpful, but not safety-critical
System decisions don’t collapse without it
Fallback behavior is less urgent than for amount_to_old_balance_ratio 

In [66]:
hit = X_stage2['amount_to_old_balance_ratio'].max()
print("hit:", hit)

hit: 5860862.89


In [68]:
hit_min = X_stage2['amount_to_old_balance_ratio'].min()
print("hit_min:", hit_min)

hit_min: 0.0


In [71]:
X_stage2['amount_to_old_balance_ratio'].describe()


count    9.273450e+05
mean     9.463834e+03
std      9.164646e+04
min      0.000000e+00
25%      7.593444e-01
50%      1.866109e+00
75%      4.766307e+00
max      5.860863e+06
Name: amount_to_old_balance_ratio, dtype: float64

In [73]:
import os
os.getcwd()

'C:\\Users\\hp\\OneDrive\\Desktop\\fraud-detection-project\\notebooks'

In [74]:
joblib.dump(rf_stage2, "C:\\Users\\hp\\OneDrive\\Desktop\\fraud-detection-project\\models\\rf_stage2.joblib")

['C:\\Users\\hp\\OneDrive\\Desktop\\fraud-detection-project\\models\\rf_stage2.joblib']